# 04 — Testing & Export

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from ml_utils import get_similar_cities as _get_similar_cities

merged = pd.read_csv("merged_final.csv")
city_stats = pd.read_csv("city_stats_stage2.csv")
district_stats = pd.read_csv("district_stats_stage2.csv")
kmeans = joblib.load("kmeans_model.pkl")
scaler = joblib.load("scaler.pkl")

def get_similar_cities(city_name, n=3):
    return _get_similar_cities(merged, city_name, n)

print("Loaded successfully:", merged.shape)

Loaded successfully: (44, 11)


## 12. analyze_location()

In [2]:
def analyze_location(city=None, postal_code=None, district=None):
    # Path A: city name provided
    if city and str(city).strip():
        city_clean = str(city).strip().title()
        row = merged[merged["city_clean"] == city_clean]
        note = None
        if row.empty:
            partial = merged[merged["city_clean"].str.startswith(city_clean)]
            if not partial.empty:
                row = partial.sort_values("listing_count", ascending=False).iloc[[0]]
                note = f"Showing results for '{row.iloc[0]['city_clean']}' (closest match to '{city}')."
            else:
                return {"status": "not_found", "message": f"No data available for '{city}'."}
        row = row.iloc[0]
        if note is None and postal_code:
            try:
                p = int(postal_code)
                if pd.notna(row["postcode"]) and int(row["postcode"]) != p:
                    note = f"Note: we have {row['city_clean']} listed under postal code {int(row['postcode'])}, not {p}."
            except (TypeError, ValueError):
                pass
        return _format_result(row["city_clean"], row, note, level="city")

    # Path B: postal code only
    if postal_code:
        try:
            p = int(postal_code)
        except (TypeError, ValueError):
            return {"status": "error", "message": "Invalid postal code."}
        row = merged[merged["postcode"] == p]
        if row.empty:
            return {"status": "not_found", "message": f"No data available for postal code '{postal_code}'."}
        row = row.iloc[0]
        note = f"Showing results for {row['city_clean']} (matched from postal code {p})."
        return _format_result(row["city_clean"], row, note, level="city")

    # Path C: district only
    if district and str(district).strip():
        d_clean = str(district).strip().title()
        row = district_stats[district_stats["district"].str.title() == d_clean]
        if row.empty:
            return {"status": "not_found", "message": f"No data available for district '{district}'."}
        row = row.iloc[0]
        note = f"District-level estimate for {row['district']} (city-level data not available)."
        return _format_result(row["district"], row, note, level="district")

    return {"status": "error", "message": "Please enter a city, postal code, or district."}


def _format_result(name, row, note, level):
    result = {
        "status": "ok",
        "location": name,
        "level": level,
        "cluster": row["cluster_label"],
        "avg_price_per_sqft": round(row["avg_price_per_sqft"], 2),
        "listing_count": int(row["listing_count"]),
        "note": note
    }
    if level == "city":
        result["postal_code"] = int(row["postcode"]) if pd.notna(row["postcode"]) else None
        result["yoy_trend_pct"] = row.get("yoy_trend_pct") if pd.notna(row.get("yoy_trend_pct", np.nan)) else None
        result["similar_cities"] = get_similar_cities(name)
    return result

### Tests

In [ ]:
import json as _json

tests = [
    {"city": "Kadawatha"},
    {"postal_code": 11850},               
    {"district": district_stats['district'].iloc[0]},  
    {"city": "Jaffna"},                
    {},                             
]

results = [analyze_location(**test) for test in tests]

assert results[0]["status"] == "ok" and results[0]["location"] == "Kadawatha"
assert results[1]["status"] == "ok" and results[1]["location"] == "Kadawatha"
assert results[2]["status"] == "ok" and results[2]["level"] == "district"
assert results[3]["status"] == "not_found"
assert results[4]["status"] == "error"

for test, result in zip(tests, results):
    print(f"--- Input: {test} ---")
    print(_json.dumps(result, indent=2, default=str))
    print()

print("All analyze_location tests passed.")

--- Input: {'city': 'Kadawatha'} ---
{
  "status": "ok",
  "location": "Kadawatha",
  "level": "city",
  "cluster": "Budget & Rising",
  "avg_price_per_sqft": 22184.74,
  "listing_count": 4,
  "note": null,
  "postal_code": 11850,
  "yoy_trend_pct": null,
  "similar_cities": [
    "Mount Lavinia",
    "Athurugiriya",
    "Kottawa"
  ]
}

--- Input: {'postal_code': 11850} ---
{
  "status": "ok",
  "location": "Kadawatha",
  "level": "city",
  "cluster": "Budget & Rising",
  "avg_price_per_sqft": 22184.74,
  "listing_count": 4,
  "note": "Showing results for Kadawatha (matched from postal code 11850).",
  "postal_code": 11850,
  "yoy_trend_pct": null,
  "similar_cities": [
    "Mount Lavinia",
    "Athurugiriya",
    "Kottawa"
  ]
}

--- Input: {'district': 'Colombo'} ---
{
  "status": "ok",
  "location": "Colombo",
  "level": "district",
  "cluster": "Fast Growing",
  "avg_price_per_sqft": 40387.24,
  "listing_count": 4503,
  "note": "District-level estimate for Colombo (city-level data

## 13. Export

In [ ]:
# City-level cluster data
city_output = []
for _, row in merged.iterrows():
    trend_row = city_stats[city_stats["city_clean"] == row["city_clean"]]
    trend = trend_row["yoy_trend_pct"].iloc[0] if not trend_row.empty else None
    city_output.append({
        "city": row["city_clean"],
        "postal_code": int(row["postcode"]) if pd.notna(row["postcode"]) else None,
        "cluster": row["cluster_label"],
        "avg_price_per_sqft": round(row["avg_price_per_sqft"], 2),
        "listing_count": int(row["listing_count"]),
        "yoy_trend_pct": trend if pd.notna(trend) else None,
        "similar_cities": get_similar_cities(row["city_clean"])
    })
with open("city_clusters.json", "w") as f:
    json.dump(city_output, f, indent=2)
print(f"Saved {len(city_output)} cities -> city_clusters.json")

# Map data (lat/long)
map_output = [{
    "city": row["city_clean"], "lat": float(row["latitude"]), "lng": float(row["longitude"]),
    "cluster": row["cluster_label"], "avg_price_per_sqft": round(row["avg_price_per_sqft"], 2),
    "listing_count": int(row["listing_count"])
} for _, row in merged.iterrows() if pd.notna(row["latitude"])]
with open("city_map_data.json", "w") as f:
    json.dump(map_output, f, indent=2)
print(f"Saved {len(map_output)} cities -> city_map_data.json")

# District-level cluster data 
district_output = [{
    "district": row["district"], "cluster": row["cluster_label"],
    "avg_price_per_sqft": round(row["avg_price_per_sqft"], 2), "listing_count": int(row["listing_count"])
} for _, row in district_stats.iterrows()]
with open("district_clusters.json", "w") as f:
    json.dump(district_output, f, indent=2)
print(f"Saved {len(district_output)} districts -> district_clusters.json")

# Postal code -> city reverse lookup (NEW, supports postal-code-only search)
postal_lookup = merged[["postcode", "city_clean"]].dropna(subset=["postcode"])
postal_output = [{"postal_code": int(r["postcode"]), "city": r["city_clean"]} for _, r in postal_lookup.iterrows()]
with open("postal_to_city.json", "w") as f:
    json.dump(postal_output, f, indent=2)
print(f"Saved {len(postal_output)} postal code mappings -> postal_to_city.json")

Saved 44 cities -> city_clusters.json
Saved 44 cities -> city_map_data.json
Saved 6 districts -> district_clusters.json
Saved 44 postal code mappings -> postal_to_city.json


In [5]:
# Save trained model artifacts and final tables
joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(scaler, "scaler.pkl")
merged.to_csv("final_city_data.csv", index=False)
district_stats.to_csv("final_district_data.csv", index=False)

for f in ["kmeans_model.pkl", "scaler.pkl", "final_city_data.csv", "final_district_data.csv",
          "city_clusters.json", "city_map_data.json", "district_clusters.json", "postal_to_city.json"]:
    print(f"{f}: {'saved' if os.path.exists(f) else 'MISSING'}")

kmeans_model.pkl: saved
scaler.pkl: saved
final_city_data.csv: saved
final_district_data.csv: saved
city_clusters.json: saved
city_map_data.json: saved
district_clusters.json: saved
postal_to_city.json: saved


## 14. Limitations

- The model groups locations using average advertised apartment price per square foot; it does not estimate final transaction prices.
- K-means is trained on one feature only. Listing count and year-on-year trend are supplementary outputs and do not influence cluster assignment.
- The cluster names are presentation labels assigned after training and should not be interpreted as model-learned evidence of growth or stability.
- A minimum of three listings is required for city and district averages, so low-volume locations may remain sensitive to individual advertisements.
- The trend proxy compares 2022 and 2023 averages and is returned only when both years contain at least three listings.
- Three unmatched locations use manually supplied postal codes and coordinates; these values should be independently verified before production use.
- The exported JSON is a precomputed snapshot consumed by the React application; the trained model is not executed live in the browser or on the blockchain.